## Gold Layer: Transactional Fact Sales

In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
import dlt
from pyspark.sql.functions import col

@dlt.table(
    name="fact_sales",
    comment="Gold Layer: Central Sales Fact table capturing validated purchase transactions",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.zOrderCols": "event_time, product_id",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true",
        "delta.logRetentionDuration": "interval 30 days", # Required for Time Travel
        "delta.deletedFileRetentionDuration": "interval 7 days", # For VACUUM policy
        "delta.enableChangeDataFeed": "true" # Enables physical table features
    }
)
def fact_sales():
    """
    Fact table implementation: Filters Silver data for 'purchase' events 
    to drive revenue and conversion analytics
    """
    return (
        # 1. Incremental read from the validated Silver layer
        dlt.read_stream("events_cleaned")
        
        # 2. Business Logic: Isolation of revenue-generating transactions
        .filter(col("event_type") == "purchase")
        
        # 3. Dimensional Selection: Core metrics and Foreign Keys for Star Schema
        .select(
            "user_id",       # FK to User Dimension
            "product_id",    # FK to Product Dimension
            "event_time",    # Temporal dimension reference
            "price",         # Quantitative revenue metric
            "user_session",   # Behavioral grouping key
            "event_type"
        )
    )